# 04 | Review friction and first-session risks

**Author: Chanakya**

Use public reviews to discover product friction, not to estimate the prevalence of problems among subscribers. Store audiences, collection windows and review-update behavior differ. No names or original review IDs are exported.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/manifests/release.json').exists())
sys.path.insert(0, str(ROOT))
from src.analysis_common import *
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
rng = np.random.default_rng(CFG['seed'])
print('Offline inputs:', CFG['raw_release'], '| Author: Chanakya')
import src.analysis_common as shared
shared.ACTIVE_NOTEBOOK='04_review_friction'
shared.ACTIVE_SOURCES=['fancode_ios_reviews_page1', 'fancode_ios_reviews_page10', 'fancode_ios_reviews_page2', 'fancode_ios_reviews_page3', 'fancode_ios_reviews_page4', 'fancode_ios_reviews_page5', 'fancode_ios_reviews_page6', 'fancode_ios_reviews_page7', 'fancode_ios_reviews_page8', 'fancode_ios_reviews_page9']

## 1. Parse preserved review responses and apply the registered window
Deduplicate by store and source review ID, retaining the latest encountered snapshot. Use source timestamps in Asia/Kolkata. The same person may appear across stores. Text stays in a local processed file and must be reviewed before redistribution.

In [ ]:
from google_play_scraper.constants.regex import Regex
from google_play_scraper.constants.element import ElementSpecs
reviews={}
for log in [json.loads(x) for x in (ROOT/'data/manifests/review_collection_log.jsonl').read_text().splitlines()]:
 if str(log['http_status'])!='200':continue
 payload=json.loads(Regex.REVIEWS.findall((ROOT/log['file']).read_text())[0]);body=json.loads(payload[0][2])
 for item in body[0] if body and body[0] else []:
  rid=ElementSpecs.Review['reviewId'].extract_content(item);key=log['app_id']+':'+rid
  reviews[key]=dict(review_key=hashlib.sha256(key.encode()).hexdigest()[:20],store='Android mobile' if log['app_id']=='com.dream11sportsguru' else 'Android TV',date=pd.Timestamp(item[5][0],unit='s',tz='UTC').tz_convert('Asia/Kolkata'),rating=ElementSpecs.Review['score'].extract_content(item),text=ElementSpecs.Review['content'].extract_content(item) or '',source_file=log['file'])
for sid in SOURCES:
 if not sid.startswith('fancode_ios_reviews_page'):continue
 entries=rawjson(sid)['feed'].get('entry',[])
 if isinstance(entries,dict):entries=[entries]
 for e in entries:
  if 'im:rating' not in e:continue
  key='iOS:'+e['id']['label'];reviews[key]=dict(review_key=hashlib.sha256(key.encode()).hexdigest()[:20],store='iOS',date=pd.Timestamp(e['updated']['label']).tz_convert('Asia/Kolkata'),rating=int(e['im:rating']['label']),text=e.get('title',{}).get('label','')+' '+e['content']['label'],source_file=SOURCES[sid]['file'])
r=pd.DataFrame(reviews.values());r=r[(r.date>=pd.Timestamp('2025-01-01',tz='Asia/Kolkata'))&(r.date<pd.Timestamp('2026-09-13',tz='Asia/Kolkata'))].copy()
r['text']=r.text.str.replace(r'https?://\S+|\S+@\S+|\b\d{10,}\b','[redacted]',regex=True);r['month']=r.date.dt.strftime('%Y-%m');r['year']=r.date.dt.year;r['low_rating']=r.rating<=2
lex=json.loads((ROOT/'analysis_config/review_lexicon.json').read_text())
for name,pattern in {**lex['patterns'],**{'sport_'+k:v for k,v in lex['sport_patterns'].items()}}.items():r[name]=r.text.str.contains(pattern,case=False,regex=True,na=False)
table(r,'review_features_local_only',True)
counts=r.groupby('store').agg(reviews=('review_key','size'),mean_rating=('rating','mean'),low_rating_share=('low_rating','mean'),tennis_mentions=('sport_tennis','sum'),first_date=('date','min'),last_date=('date','max')).reset_index();display(table(counts,'04_store_coverage'))

## 2. Issue mentions, not automatically complaints
The lexicon is deliberately transparent and frozen in analysis_config. “Pass” or “quality” can appear in praise. Report mention rates inside low-rated reviews separately, and never convert mentions into churn or lost revenue. Wilson intervals describe binomial uncertainty conditional on this review sample, not selection bias.

In [ ]:
from statsmodels.stats.proportion import proportion_confint
mention=[]
for store,g in r.groupby('store'):
 for subset,h in [('All',g),('Rating 1–2',g[g.low_rating])]:
  for topic in lex['patterns']:
   k=int(h[topic].sum());n=len(h);lo,hi=proportion_confint(k,n,method='wilson') if n else (np.nan,np.nan)
   mention.append(dict(store=store,subset=subset,topic=topic,mentions=k,review_denominator=n,share=k/n if n else np.nan,wilson_low=lo,wilson_high=hi))
mention=pd.DataFrame(mention);display(table(mention,'04_issue_mentions'))
g=mention[(mention.store=='Android mobile')&(mention.subset=='Rating 1–2')].sort_values('share')
plt.figure(figsize=(9,4));plt.barh(g.topic.str.replace('_',' '),g.share*100);plt.errorbar(g.share*100,range(len(g)),xerr=[(g.share-g.wilson_low)*100,(g.wilson_high-g.share)*100],fmt='none',ecolor='black',capsize=3);plt.xlabel('Percent of low-rated mobile reviews');plt.title('First-session friction deserves a place in the growth strategy');fig('04_friction_mentions','Heuristic topic mentions, overlapping categories. Conditional Wilson intervals do not remove review selection bias.')
monthly=r.groupby(['store','month']).agg(reviews=('review_key','size'),rating=('rating','mean'),low_share=('low_rating','mean'),tennis_mentions=('sport_tennis','sum')).reset_index();table(monthly,'04_monthly_review_diagnostics')
figx,ax=plt.subplots(2,1,figsize=(11,6),sharex=True)
for store,g in monthly.groupby('store'):
 ax[0].plot(pd.to_datetime(g.month),g.rating,marker='.',label=store);ax[1].plot(pd.to_datetime(g.month),g.reviews,label=store)
ax[0].set(ylabel='Mean review rating',title='Review composition changes over time');ax[0].legend();ax[1].set(ylabel='Reviews collected');fig('04_review_time_series','Public storefront sample, not a before/after causal ATP-launch evaluation. iOS is a latest-500 window.')
# Fairer calendar alignment: only Jan-Aug in both years, mobile store only.
matched=r[(r.store=='Android mobile')&(r.date.dt.month<=8)].groupby('year').agg(n=('review_key','size'),low_share=('low_rating','mean'),tennis_mentions=('sport_tennis','sum'),access_mentions=('stream_access','mean')).reset_index();display(table(matched,'04_matched_month_comparison'))

## 3. Exploratory themes with a chronological holdout
Fit TF-IDF and six nonnegative matrix factorization components on 2025 mobile reviews only. Transform 2026 without refitting. This checks whether a vocabulary is still usable, not whether topics are true customer segments. NMF components are overlapping latent themes. No cluster is assigned a market size.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
train=r[(r.store=='Android mobile')&(r.year==2025)&(r.text.str.len()>=25)]
hold=r[(r.store=='Android mobile')&(r.year==2026)&(r.text.str.len()>=25)]
vectorizer=TfidfVectorizer(stop_words='english',min_df=5,max_df=.8,max_features=2500,ngram_range=(1,2),token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b')
X=vectorizer.fit_transform(train.text);Y=vectorizer.transform(hold.text)
nmf=NMF(n_components=6,init='nndsvda',random_state=CFG['seed'],max_iter=700)
W=nmf.fit_transform(X);H=nmf.transform(Y);words=vectorizer.get_feature_names_out()
topics=pd.DataFrame([dict(component=k,top_terms=', '.join(words[np.argsort(row)[-10:][::-1]]),train_documents=len(train),holdout_documents=len(hold),mean_train_weight=W[:,k].mean(),mean_holdout_weight=H[:,k].mean()) for k,row in enumerate(nmf.components_)])
display(table(topics,'04_exploratory_themes'))
print('Holdout documents with no training vocabulary:',int((Y.getnnz(axis=1)==0).sum()))
# Export only aggregate sport-mention diagnostics, not claims of sport-specific customer prevalence.
sports=pd.DataFrame([dict(sport=k,mentions=int(r['sport_'+k].sum()),low_rated_mentions=int((r['sport_'+k]&r.low_rating).sum())) for k in lex['sport_patterns']]);display(table(sports,'04_sport_mention_counts'))
check('04_reviews',{'deduplicated_review_keys':r.review_key.is_unique,'mobile_window_count':int(counts.set_index('store').loc['Android mobile','reviews'])==14665,'tv_window_count':int(counts.set_index('store').loc['Android TV','reviews'])==64,'ios_window_count':int(counts.set_index('store').loc['iOS','reviews'])==500,'ratings_valid':bool(r.rating.between(1,5).all()),'train_before_holdout':train.date.max()<hold.date.min()})
report('04_review_findings','Prioritize a measurable pay-to-play reliability and entitlement journey. Topic counts are mentions, not adjudicated complaints, and no causal pre/post claim is made. NMF is exploratory with a chronological holdout. Sport labels are keyword signals, not customer identities. Raw and row-level review text remain local-only. Formal classifier validation and subscriber-level incidence require other data.')

## Source references
These IDs resolve to the preserved bodies, URLs and capture timestamps. Derived tables also retain row-level source IDs where applicable. Case inputs refer to the supplied brief, physical PDF pages 9–14. Review source files resolve through the review collection log. Scenario parameters are in analysis_config.

In [ ]:
references=source_table(['fancode_ios_reviews_page1', 'fancode_ios_reviews_page10', 'fancode_ios_reviews_page2', 'fancode_ios_reviews_page3', 'fancode_ios_reviews_page4', 'fancode_ios_reviews_page5', 'fancode_ios_reviews_page6', 'fancode_ios_reviews_page7', 'fancode_ios_reviews_page8', 'fancode_ios_reviews_page9'])
display(table(references,'04_source_references'))